In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")

DATA_DIR = Path("../data/interim")
TRAIN_PATH = DATA_DIR / "train_merged.parquet"


# 03 - EDA: Fitur Identitas

## Konteks
Tabel identitas menyimpan data fingerprinting perangkat dan browser — tapi hanya tersedia untuk sekitar 25% transaksi. Ini penting karena kita perlu tahu kapan data ini ada dan apa artinya kalau tidak ada.

## Tujuan
- Cek seberapa banyak transaksi yang punya identity data
- Lihat apakah kehadiran identity data berkorelasi dengan fraud
- Eksplorasi DeviceType dan DeviceInfo
- Lihat distribusi dan korelasi fitur id_01 sampai id_38

In [ ]:
train = pd.read_parquet(TRAIN_PATH)
print(f'Data loaded: {train.shape}')


In [ ]:
print(f'Total columns: {len(train.columns)}')
print(f'Identity columns : {len([x for x in train.columns if x.startswith("id_")])}')


In [ ]:
# kolom identitas
id_cols = [c for c in train.columns if c.startswith('id_')]
device_cols = ['DeviceType', 'DeviceInfo']

# cek apakah ada data identitas di tiap baris
identity = train[id_cols + device_cols].notna().any(axis=1)

print('Identity Coverage')
print(f'Total Transaction : {len(train):,}')
print(f'   With identity: {identity.sum():,} ({identity.mean()*100:.1f}%)')
print(f'   Without identity: {(~identity).sum():,} ({(~identity).mean()*100:.1f}%)')

### Coverage: Seberapa Banyak Transaksi Punya Identity Data?

In [ ]:
# fraud rate berdasarkan kehadiran data identitas

fraud_id = train.loc[identity, 'isFraud'].mean() * 100
fraud_without_id = train.loc[~identity, 'isFraud'].mean() * 100

print('Fraud rate based on identity data presence')
print(f'with identity    : {fraud_id:.2f}%')
print(f'without identity : {fraud_without_id:.2f}%')

# visualisasi perbandingan fraud rate
fig, ax = plt.subplots(figsize=(8,6))
ax.bar(['With Identity', 'Without Identity'], [fraud_id, fraud_without_id])
ax.set_ylabel('Fraud Rate (%)')
ax.set_title('Fraud Rate: With vs Without Identity Data')
ax.axhline(y=train['isFraud'].mean() * 100, linestyle='--', label='Overall Average')
ax.legend()
plt.tight_layout()
plt.show()

## DeviceType

In [ ]:
print('DeviceType Distribution:')
print(train['DeviceType'].value_counts(dropna=False))

device_fraud = train.groupby('DeviceType', dropna=False)['isFraud'].agg(['mean', 'count'])
device_fraud['mean'] *= 100

print('Fraud rate by DeviceType:')
print(device_fraud)


In [ ]:
# lihat device info yang paling banyak muncul dan fraud rate-nya
# ambil top 15 agar chart tidak terlalu ramai
top_device = train['DeviceInfo'].value_counts().head(15)

print('Top 15 Most Frequent DeviceInfo:')
print(top_device)

# fraud rate per device info 
fraud_by_device = train.groupby('DeviceInfo')['isFraud'].mean() * 100
fraud_by_device = fraud_by_device.loc[top_device.index]

fig, ax = plt.subplots(figsize=(10,6))
fraud_by_device.sort_values(ascending=False).plot(kind='bar', ax=ax)
ax.set_title('Fraud Rate per Device Info (Top 15)')
ax.set_ylabel('Fraud Rate (%)')
ax.axhline(y=train['isFraud'].mean() * 100, color='red', linestyle='--', label='Overal Rate')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Fitur ID (id_01 – id_38)

In [ ]:
id_cols = [f'id_{i:02d}' for i in range(1,39)]
id_cols = [c for c in id_cols if c in train.columns]

print(f'Total ID features: {len(id_cols)} columns')

# hitung missing rate dan jumlah nilai unik tiap fitur ID 
id_stats = []
for col in id_cols:
    id_stats.append({
        'columns' : col,
        'missing_pct' : train[col].isnull().sum() / len(train) * 100,
        'nunique' : train[col].nunique(),
        'dtype' : str(train[col].dtype)
    })

id_df = pd.DataFrame(id_stats)
print(id_df.to_string())

In [ ]:
# korelasi fitur ID numerik terhadap target isFraud

numeric_id_cols = train[id_cols].select_dtypes(include=[np.number]).columns.tolist()

if numeric_id_cols:
    id_corr = train[numeric_id_cols + ['isFraud']].corr()['isFraud'].drop('isFraud').sort_values()

    print('Correlation of numerical ID features with isFraud:')
    print(id_corr)

    fig, ax = plt.subplots(figsize=(10,6))
    id_corr.plot(kind='barh', ax = ax)
    ax.set_title('Correlation of ID Features with isFraud')
    plt.xlabel('Correlation')
    ax.axvline(x=0, color='black', linestyle='-')
    plt.tight_layout()
    plt.show()
    